# MOBBLOG INTELLIGENCE — Exploración inicial de datos (EDA)

**Objetivo:** comprender el dataset de referencia de X Education antes de preparar variables o entrenar modelos.

**Importante:** este archivo es un conjunto experimental externo; no representa el historial comercial de MOBBLOG Digital. No entrenamos ningún modelo en esta etapa.

## Cómo ejecutar
1. Coloca `Leads.csv` en la misma carpeta que este notebook.
2. Abre el notebook con Jupyter o Google Colab.
3. Ejecuta las celdas en orden.

## 1. Carga de librerías y datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('Leads.csv')
print(f'Dimensiones: {df.shape[0]:,} filas × {df.shape[1]} columnas')
display(df.head())

## 2. Estructura y tipos de datos

In [ ]:
display(df.info())
display(df.describe(include='all').T)

## 3. Calidad: duplicados y valores faltantes

In [ ]:
duplicados = df.duplicated().sum()
faltantes = df.isna().sum().sort_values(ascending=False)
resumen_faltantes = pd.DataFrame({
    'faltantes': faltantes,
    'porcentaje': (faltantes / len(df) * 100).round(2)
})
print('Filas duplicadas completas:', duplicados)
display(resumen_faltantes[resumen_faltantes['faltantes'] > 0])

## 4. Variable objetivo: `Converted`

`Converted` es la variable objetivo del dataset: 1 indica conversión y 0 indica que no hubo conversión. Esta distribución describe únicamente este conjunto de datos.

In [ ]:
conteo = df['Converted'].value_counts().sort_index()
porcentaje = (conteo / len(df) * 100).round(2)
display(pd.DataFrame({'registros': conteo, 'porcentaje': porcentaje}))
conteo.plot(kind='bar', rot=0)
plt.title('Distribución de Converted')
plt.xlabel('Converted (0 = no, 1 = sí)')
plt.ylabel('Número de registros')
plt.tight_layout()
plt.show()

## 5. Revisión de variables categóricas

In [ ]:
categoricas = df.select_dtypes(include=['object']).columns
for col in categoricas:
    print(f'\\n--- {col} | valores únicos: {df[col].nunique(dropna=True)} ---')
    display(df[col].value_counts(dropna=False).head(10).to_frame('frecuencia'))

## 6. Revisión de variables numéricas

Los valores extremos no se eliminan automáticamente. Primero se inspeccionan y luego se decide su tratamiento con justificación.

In [ ]:
numericas = df.select_dtypes(include=np.number).columns
display(df[numericas].describe().T)
for col in numericas:
    if col != 'Converted':
        df[col].plot(kind='hist', bins=30, title=f'Distribución: {col}')
        plt.xlabel(col)
        plt.tight_layout()
        plt.show()

## 7. Alertas para la preparación posterior

- Investigar valores como `Select`, que pueden representar una opción no seleccionada y no un dato real.
- No imputar ni eliminar columnas sin revisar su significado.
- Excluir identificadores como `Prospect ID` y `Lead Number` de los predictores, salvo justificación metodológica.
- Evitar fuga de información (*data leakage*): solo usar variables disponibles en el instante en que se estimaría la conversión.
- No interpretar asociaciones como causalidad.
- Consultar el diccionario de datos antes de cerrar decisiones de limpieza.

## 8. Conclusión de esta etapa

Este notebook realiza exploración descriptiva. No entrena modelos, no predice prospectos y no demuestra capacidad de conversión. Las decisiones de preparación se documentarán en una etapa posterior.